# `module.exports` in Node.js

In Node.js, `module.exports` is the built-in object that a file returns when it is imported into another file using the `require()` function. It acts as the public interface for a file, allowing you to expose functions, objects, classes, or values while keeping internal logic private.

Anything you *don't* attach to `module.exports` stays private to that file — there is no `public`/`private` keyword in JavaScript, and this is the mechanism that replaces it.

## Core Patterns for Exporting Code

### 1. Named Exports (Exporting Multiple Items)

Attach multiple properties directly to the `module.exports` object.

```javascript
// math.js
const add = (a, b) => a + b;
const subtract = (a, b) => a - b;

// Attach functions as properties
module.exports.add = add;
module.exports.subtract = subtract;
```

When importing, use destructuring to extract the specific functions:

```javascript
// app.js
const { add, subtract } = require('./math');

console.log(add(5, 3));      // Output: 8
console.log(subtract(5, 3)); // Output: 2
```

A cleaner variant most codebases prefer — declare everything normally, then export once at the bottom using object shorthand:

```javascript
// math.js
const add = (a, b) => a + b;
const subtract = (a, b) => a - b;
const PI = 3.14159;

module.exports = { add, subtract, PI };
```

This keeps the file's public surface in one visible place instead of scattered across the file.

### 2. Default Export (Exporting a Single Item)

Overriding `module.exports` completely replaces the exported value with a single function, object, or class.

```javascript
// logger.js
function logMessage(message) {
    console.log(`[LOG]: ${message}`);
}

// Reassign the whole object
module.exports = logMessage;
```

When importing a single default export, you can name the variable whatever you like:

```javascript
// app.js
const customLog = require('./logger');
customLog('System running smoothly'); // Output: [LOG]: System running smoothly
```

The same works for classes, which is the conventional pattern for one-class-per-file:

```javascript
// User.js
class User { /* ... */ }
module.exports = User;

// app.js
const User = require('./User');
```

## The Difference: `module.exports` vs `exports`

Node.js provides a shorthand variable named `exports`. Under the hood, Node wraps your file in a function where `exports` is simply a reference pointing at `module.exports`:

```javascript
(function (exports, require, module, __filename, __dirname) {
  // your file's code lives here
});
```

This wrapper is also where `require`, `__filename`, and `__dirname` come from — they aren't globals, they're arguments. That's why they don't exist in ES modules.

Because `exports` is just a second name for the same object:

- **`exports.key = value` works** — adding properties modifies the original object.
- **`exports = value` fails** — reassigning `exports` breaks its reference link to `module.exports`. Node still returns the original `module.exports` object, which remains empty.

```javascript
// ❌ THIS WILL NOT WORK
exports = function() { console.log("Hello"); };

// ✅ THIS WILL WORK
module.exports = function() { console.log("Hello"); };
```

The practical rule: **just always write `module.exports`.** The shorthand saves seven characters and costs you a debugging session.

## Two Behaviours That Surprise People

### Modules are cached (they're effectively singletons)

A module's code runs **once**, the first time it is required. Every later `require()` of the same resolved path returns the exact same object from `require.cache`.

```javascript
// counter.js
let count = 0;
module.exports = { increment: () => ++count };

// a.js and b.js both require('./counter')
// They share one counter — not two.
```

This is useful on purpose (database connection pools, config objects) and a trap by accident (mutable shared state between unrelated files). It's also why tests sometimes need `jest.resetModules()`.

### `require()` is synchronous

It blocks while reading and evaluating the file. That's fine at the top of a file during startup, but calling `require()` conditionally inside a hot request handler means blocking the event loop on disk I/O.

## How Node Resolves a Path

| You write | Node looks for |
| --- | --- |
| `require('node:fs')` | A built-in core module (the `node:` prefix makes this explicit and unambiguous) |
| `require('./math')` | A relative file: `./math.js`, then `./math.json`, then `./math/index.js` |
| `require('express')` | `node_modules/express`, walking up parent directories until found |
| `require('/abs/path')` | That absolute path |

For a package, the entry point comes from the `"main"` or `"exports"` field in its `package.json`.

## Circular Dependencies

If `a.js` requires `b.js` and `b.js` requires `a.js`, Node doesn't crash — it returns whatever `a.js` had exported *so far*, which is often an empty object. The result is a confusing `undefined is not a function` at runtime rather than a clear error. If you hit this, the fix is usually to extract the shared piece into a third module rather than to work around it.

## The Modern Alternative: ES Modules

CommonJS (`require` / `module.exports`) is Node's original system. ES Modules (`import` / `export`) are the language standard and the default choice for new projects. You'll meet both, because a decade of npm packages and tutorials use CommonJS.

```javascript
// CommonJS                          // ES Modules
module.exports = { add };            export { add };
module.exports = logMessage;         export default logMessage;
const { add } = require('./math');   import { add } from './math.js';
const log = require('./logger');     import log from './logger.js';
```

Key practical differences:

- Enable ESM with `"type": "module"` in `package.json`, or the `.mjs` extension.
- ESM requires the **file extension** in relative imports (`'./math.js'`, not `'./math'`).
- ESM supports top-level `await`; CommonJS does not.
- ESM has no `__dirname` — use `import.meta.dirname`.
- ESM imports are hoisted and statically analysable, which is what makes tree-shaking possible.
- ESM can import CommonJS modules; CommonJS can only load ESM via dynamic `await import()`.

## Summary Table

| Approach | Implementation | Best For | Import Method |
| --- | --- | --- | --- |
| Named exports | `module.exports = { func }` | Utility libraries, multiple helpers | `const { func } = require('./file')` |
| Default export | `module.exports = func` | Single classes, primary functions | `const func = require('./file')` |